# 1 Using LLM Generated Embeddings and Cosine Similarity

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
#  1. Load Pre-computed Data 
# Load the embeddings you saved from your previous notebook.
print(" 1. Loading pre-computed embeddings and post data ")
try:
    embeddings_data = np.load('showerthoughts_embeddings.npz')
    # It's good practice to allow loading of pickled objects if needed, but be aware of security.
    # For this project, it's safe as you created the file.
    # embeddings_data = np.load('post_embeddings.npz', allow_pickle=True)
except FileNotFoundError:
    print("Error: 'showerthoughts_embeddings.npz' not found. Please run the embedding generation script first.")
    exit()

# Load the original dataframe to get post titles for presenting results.
try:
    posts_df = pd.read_csv('reddit_showerthoughts.tsv', sep='\t', on_bad_lines='skip')
    # Ensure the dataframe is clean, especially the title column
    posts_df = posts_df.dropna(subset=['title'])
    posts_df = posts_df.reset_index(drop=True)
except FileNotFoundError:
    print("Error: 'reddit_showerthoughts.tsv' not found.")
    exit()

# Extract the arrays from the loaded file
post_ids = embeddings_data['ids']
title_embeddings = embeddings_data['title_embed']
content_embeddings = embeddings_data['content_embed']

print(f"Loaded {len(post_ids)} post IDs and their embeddings.")
print("-" * 30)

 1. Loading pre-computed embeddings and post data 
Loaded 95084 post IDs and their embeddings.
------------------------------


In [8]:
# 2. Combine Title and Selftext Embeddings
# A simple and effective way to combine them is to average them.
# This creates a single vector representing the overall meaning of the post.
print("\n--- 2. Combining title and selftext embeddings ---")
# We use a weighted average. Let's give the title slightly more importance.
# You can experiment with these weights.
title_weight = 0.8
content_weight = 0.2
# combined_embeddings = (title_weight * title_embeddings) + (content_weight * content_embeddings)
combined_embeddings = title_embeddings

print(f"Combined embeddings created with shape: {combined_embeddings.shape}")
print("-" * 30)


--- 2. Combining title and selftext embeddings ---
Combined embeddings created with shape: (95084, 768)
------------------------------


In [9]:
# 3. Simulate a User and Build a User Profile
# To test our recommender, let's pretend to be a user who liked a few specific posts.
# We will find these posts in our dataframe and use their embeddings to create a user profile.
print("\n--- 3. Simulating a user and building their profile ---")

# Let's pick a few posts based on their titles.
liked_post_titles = [
    "Muffins are grown up cupcakes",
    "Guns are like night lights for adults",
    "A drug dealer is basically a murderer."
]

# Find the indices of these posts in our dataframe
liked_indices = posts_df[posts_df['title'].isin(liked_post_titles)].index.tolist()

if not liked_indices:
    print("Could not find the example liked posts in the dataset. Picking 3 random posts instead.")
    liked_indices = np.random.choice(len(posts_df), 3, replace=False).tolist()

print("Simulated user liked the following posts:")
for idx in liked_indices:
    print(f"  - {posts_df.iloc[idx]['title']}")

# Build the user profile by averaging the embeddings of the posts they liked.
user_profile_vector = np.mean(combined_embeddings[liked_indices], axis=0)
# Reshape for sklearn's cosine_similarity function
user_profile_vector = user_profile_vector.reshape(1, -1)
print(f"\nUser profile vector created with shape: {user_profile_vector.shape}")
print("-" * 30)


--- 3. Simulating a user and building their profile ---
Simulated user liked the following posts:
  - Guns are like night lights for adults
  - A drug dealer is basically a murderer.
  - Muffins are grown up cupcakes

User profile vector created with shape: (1, 768)
------------------------------


In [10]:
# 4. Find and Rank Recommendations
# Now, we calculate the similarity between our user's profile and ALL posts in the dataset.
print("\n--- 4. Calculating similarity and finding recommendations ---")
# Use cosine similarity to find the most similar posts
similarity_scores = cosine_similarity(user_profile_vector, combined_embeddings)

# The result is a 2D array, so we flatten it to a 1D array of scores
similarity_scores = similarity_scores.flatten()

# Get the indices of the top N most similar posts
# We use argsort to get indices of sorted values, then reverse them for descending order.
N = 10
# We add len(liked_indices) to N because the most similar posts will be the ones the user already liked.
top_n_indices = similarity_scores.argsort()[-(N + len(liked_indices)) :][::-1]

# Filter out the posts the user has already seen
recommendation_indices = [idx for idx in top_n_indices if idx not in liked_indices]

print(f"\nTop {N} recommendations for our user:")
for i, idx in enumerate(recommendation_indices[:N]):
    # Get the post title from the original dataframe
    rec_title = posts_df.iloc[idx]['title']
    rec_score = similarity_scores[idx]
    print(f"{i+1}. (Score: {rec_score:.4f}) {rec_title}")


--- 4. Calculating similarity and finding recommendations ---

Top 10 recommendations for our user:
1. (Score: 0.5489) A gun that shoots heroin can have two very different meanings
2. (Score: 0.5316) If you deal drugs, you're a drug dealer
3. (Score: 0.5300) A cupcake is just a muffin in disguise.
4. (Score: 0.5240) Carrying a gun everywhere is the adult version of having a security blanket.
5. (Score: 0.5217) Guns are magic wands that turn people into corpses
6. (Score: 0.5184) Pistols are baby rifles
7. (Score: 0.5159) A pharmacist is literally a drug dealer
8. (Score: 0.5074) Drug dealers go from being monsters to being your best friend as you grow older
9. (Score: 0.4994) Muffins ought to be called breakfast cupcakes
10. (Score: 0.4955) Pharmacists are master drug dealers.


# 2 Content-base System with TFIDF as Vectorizer

In [14]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# 1. Load and Prepare Data
print("--- 1. Loading and preparing post data ---")
try:
    posts_df = pd.read_csv('reddit_showerthoughts.tsv', sep='\t', on_bad_lines='skip')
    # Ensure the dataframe is clean
    posts_df = posts_df.dropna(subset=['title'])
    # Combine title and selftext into a single content field for TF-IDF
    # We fill any missing selftext with an empty string
    posts_df['content'] = posts_df['title'] + ' ' + posts_df['selftext'].fillna('')
    posts_df = posts_df.reset_index(drop=True)
except FileNotFoundError:
    print("Error: 'reddit_showerthoughts.tsv' not found.")
    exit()

print(f"Loaded and prepared {len(posts_df)} posts.")
print("-" * 30)

--- 1. Loading and preparing post data ---
Loaded and prepared 95084 posts.
------------------------------


In [ ]:
# 2. Vectorize Content with TF-IDF
# TF-IDF (Term Frequency-Inverse Document Frequency) converts text into a matrix
# of token counts, where each word's importance is weighted by how frequently
# it appears in a document versus how frequently it appears across all documents.
print("\n--- 2. Vectorizing text with TF-IDF ---")
# Initialize the vectorizer
# - stop_words='english': Removes common English words (like 'the', 'a', 'is')
# - max_df=0.95: Ignores terms that appear in more than 95% of documents (too common)
# - min_df=2: Ignores terms that appear in less than 2 documents (too rare)
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_df=0.95, min_df=2)

# Fit the vectorizer to the data and transform the content into a sparse matrix
tfidf_matrix = tfidf_vectorizer.fit_transform(posts_df['content'])

print(f"Created a TF-IDF matrix with shape: {tfidf_matrix.shape}")
print("(This means we have {} posts and {} unique words in our vocabulary)".format(
    tfidf_matrix.shape[0], tfidf_matrix.shape[1]
))
print("-" * 30)


--- 2. Vectorizing text with TF-IDF ---
Created a TF-IDF matrix with shape: (95084, 23234)
(This means we have 95084 posts and 23234 unique words in our vocabulary)
------------------------------


In [ ]:
# 3. Simulate a User and Build a User Profile
# We use the same liked posts as the LLM example for a direct comparison.
print("\n--- 3. Simulating a user and building their profile ---")

# Let's pick a few posts based on their titles.
liked_post_titles = [
    "Muffins are grown up cupcakes",
    "Guns are like night lights for adults",
    "A drug dealer is basically a murderer."
]

# Find the indices of these posts in our dataframe
liked_indices = posts_df[posts_df['title'].isin(liked_post_titles)].index.tolist()

if not liked_indices:
    print("Could not find the example liked posts in the dataset. Picking 3 random posts instead.")
    liked_indices = np.random.choice(len(posts_df), 3, replace=False).tolist()

print("Simulated user liked the following posts:")
for idx in liked_indices:
    print(f"  - {posts_df.iloc[idx]['title']}")

# Build the user profile by averaging the TF-IDF vectors of the posts they liked.
# We use np.asarray to convert the resulting np.matrix to a standard np.ndarray,
# which is required by the cosine_similarity function.
user_profile_vector = np.asarray(tfidf_matrix[liked_indices].mean(axis=0))
print(f"\nUser profile vector created.")
print("-" * 30)


--- 3. Simulating a user and building their profile ---
Simulated user liked the following posts:
  - Guns are like night lights for adults
  - A drug dealer is basically a murderer.
  - Muffins are grown up cupcakes

User profile vector created.
------------------------------


In [ ]:
# 4. Find and Rank Recommendations
# Calculate the similarity between our user's profile and ALL posts in the dataset.
print("\n--- 4. Calculating similarity and finding recommendations ---")
# Use cosine similarity to find the most similar posts
# It works efficiently with the sparse matrix from TF-IDF
similarity_scores = cosine_similarity(user_profile_vector, tfidf_matrix)

# The result is a 2D array, so we flatten it to a 1D array of scores
similarity_scores = similarity_scores.flatten()

# Get the indices of the top N most similar posts
N = 10
top_n_indices = similarity_scores.argsort()[-(N + len(liked_indices)) :][::-1]

# Filter out the posts the user has already seen
recommendation_indices = [idx for idx in top_n_indices if idx not in liked_indices]

print(f"\nTop {N} recommendations for our user (using TF-IDF):")
for i, idx in enumerate(recommendation_indices[:N]):
    # Get the post title from the original dataframe
    rec_title = posts_df.iloc[idx]['title']
    rec_score = similarity_scores[idx]
    print(f"{i+1}. (Score: {rec_score:.4f}) {rec_title}")


--- 4. Calculating similarity and finding recommendations ---

Top 10 recommendations for our user (using TF-IDF):
1. (Score: 0.3841) Muffins ought to be called breakfast cupcakes
2. (Score: 0.3666) Muffins are to cupcakes as smoothies are to milkshakes
3. (Score: 0.3294) If you deal drugs, you're a drug dealer
4. (Score: 0.3278) If caffeine is a drug then Starbucks is a drug dealer.
5. (Score: 0.3027) A pharmacist is literally a drug dealer
6. (Score: 0.2987) If you kill the murderer, you become the murderer
7. (Score: 0.2932) A 14 year old can find a drug dealer but the police can't.
8. (Score: 0.2877) A pharmacist is a drug dealer with an education
9. (Score: 0.2803) muffins are just unfrosted cake
10. (Score: 0.2792) English Muffins are still called English Muffins in the UK.


# 3 Eval with Voting History - Precision@N and Recall@N

In [11]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import re

In [27]:
# 1. Load All Necessary Data
print("--- 1. Loading all data ---")
# Load the pre-computed LLM embeddings
try:
    embeddings_data = np.load('showerthoughts_embeddings.npz')
    title_embeddings = embeddings_data['title_embed']
    content_embeddings = embeddings_data['content_embed']
    # Combine them
    # combined_llm_embeddings = (0.8 * title_embeddings) + (0.2 * content_embeddings)
    combined_llm_embeddings = title_embeddings

except FileNotFoundError:
    print("Error: 'showerthoughts_embeddings.npz' not found. Please run the embedding generation script first.")
    exit()

# Load the posts and votes data
try:
    posts_df = pd.read_csv('reddit_showerthoughts.tsv', sep='\t', on_bad_lines='skip').dropna(subset=['title']).reset_index()
    # Use the specified votes file
    votes_df = pd.read_csv('44_million_votes.txt', sep='\t')
except FileNotFoundError:
    print("Error: 'reddit_showerthoughts.tsv' or '44_million_vote.txt' not found.")
    exit()

print("Data loaded successfully.")
print("-" * 30)

--- 1. Loading all data ---
Data loaded successfully.
------------------------------


In [28]:
# 2. Prepare Data for Evaluation
print("\n--- 2. Preparing data for evaluation ---")
# We now consider BOTH upvotes and downvotes as engagement signals.
# We filter out any other potential values in the VOTE column.
votes_df = votes_df[votes_df['VOTE'].isin(['upvote', 'downvote'])]

# Create a reliable index for the embedding matrix
# The embedding matrix is aligned with the cleaned posts_df. We reset its index
# to get a clean 0-to-N-1 sequence and store it in a new column.
posts_df = posts_df.reset_index(drop=True)
posts_df['embedding_idx'] = posts_df.index

# Merge votes with posts to get post details and the new embedding_idx
posts_df.rename(columns={'submission_id': 'SUBMISSION_ID'}, inplace=True)
merged_df = pd.merge(votes_df, posts_df, on='SUBMISSION_ID')

# For meaningful evaluation, focus on users with a decent number of total votes
user_vote_counts = merged_df['USERNAME'].value_counts()
active_users = user_vote_counts[user_vote_counts >= 20].index.tolist() # Increased threshold for robustness
eval_df = merged_df[merged_df['USERNAME'].isin(active_users)]

print(f"Found {len(active_users)} active users (>= 20 votes) for evaluation.")
print("-" * 30)


--- 2. Preparing data for evaluation ---
Found 3251 active users (>= 20 votes) for evaluation.
------------------------------


In [29]:
# 3. Standard Evaluation Function (Precision/Recall)
def evaluate_recommender(user_item_df, item_features_matrix, top_n=10):
    """
    Performs an offline evaluation using Precision@N and Recall@N.
    """
    user_groups = user_item_df.groupby('USERNAME')
    precisions = []
    recalls = []

    print(f"Evaluating for {len(user_groups)} users...")
    for username, group in tqdm(user_groups):
        # Split user's upvoted items into a training set and a test set
        if len(group) < 5: # Need at least a few items to train and test
            continue

        test_size = max(1, int(len(group) * 0.2))
        train_items = group.sample(frac=1, random_state=42).iloc[test_size:]
        test_items = group.sample(frac=1, random_state=42).iloc[:test_size]

        train_indices = train_items['embedding_idx'].tolist()
        test_indices = set(test_items['embedding_idx'].tolist())

        # Build user profile from the training items
        user_profile = item_features_matrix[train_indices].mean(axis=0)
        if isinstance(user_profile, np.matrix):
            user_profile = np.asarray(user_profile)
        user_profile = user_profile.reshape(1, -1)

        # Generate recommendations
        similarity_scores = cosine_similarity(user_profile, item_features_matrix).flatten()
        similarity_scores[train_indices] = -1 # Exclude items from training set

        # Get top N recommendations
        recommended_indices = set(similarity_scores.argsort()[::-1][:top_n])

        # Calculate metrics
        hits = len(recommended_indices & test_indices)

        precision_at_n = hits / top_n
        recall_at_n = hits / len(test_indices)

        precisions.append(precision_at_n)
        recalls.append(recall_at_n)

    avg_precision = np.mean(precisions) if precisions else 0
    avg_recall = np.mean(recalls) if recalls else 0
    return avg_precision, avg_recall


In [30]:
# 4. Run Evaluation for Both Models
print("\n--- 4. Running evaluations ---")

# A) Evaluate LLM-based Recommender
print("\n--- Evaluating LLM Recommender ---")
llm_precision, llm_recall = evaluate_recommender(eval_df, combined_llm_embeddings, top_n=100)
print(f"\nLLM Model Results:")
print(f"  - Average Precision@10: {llm_precision:.4f}")
print(f"  - Average Recall@10:    {llm_recall:.4f}")

# B) Evaluate TF-IDF-based Recommender
print("\n--- Evaluating TF-IDF Recommender ---")
# cleaning
def clean_text(text):
    text = text.lower()  
    text = re.sub(r"[^a-zA-Z0-9'’ –-]|(?<= )[–-] |(?<=[^a-zA-Z0-9])['’]|['’](?=[^a-zA-Z0-9])", " ", text) 
    return text
def best_preprocessor(text):
    return clean_text(text)
# First, create the TF-IDF matrix for all posts
posts_df['content'] = posts_df['title'] + ' ' + posts_df['selftext'].fillna('')
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_df=0.95, min_df=2,preprocessor=best_preprocessor)
tfidf_matrix = tfidf_vectorizer.fit_transform(posts_df['content'])

tfidf_precision, tfidf_recall = evaluate_recommender(eval_df, tfidf_matrix, top_n=10)
print(f"\nTF-IDF Model Results:")
print(f"  - Average Precision@10: {tfidf_precision:.4f}")
print(f"  - Average Recall@10:    {tfidf_recall:.4f}")



--- 4. Running evaluations ---

--- Evaluating LLM Recommender ---
Evaluating for 3251 users...


NameError: name 'tqdm' is not defined

# 4 Evaluation - Mean Reciprocal Rank

The [mean reciprocal rank](https://en.wikipedia.org/wiki/Mean_reciprocal_rank) is a metric used to evaluate the effectiveness of a ranking system, particularly in scenarios where the position of the first relevant result is crucial. It focuses on how quickly a system surfaces the first relevant item in a ranked list of results. MRR is calculated by averaging the reciprocal ranks of the first relevant result for a set of queries. 

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm # For progress bars

In [ ]:
# 1. Load All Necessary Data
print("--- 1. Loading all data ---")
# Load the pre-computed LLM embeddings
try:
    embeddings_data = np.load('showerthoughts_embeddings.npz')
    title_embeddings = embeddings_data['title_embed']
    content_embeddings = embeddings_data['content_embed']
    # Combine them
    combined_llm_embeddings = (0.6 * title_embeddings) + (0.4 * content_embeddings)
except FileNotFoundError:
    print("Error: 'showerthoughts_embeddings.npz' not found. Please run the embedding generation script first.")
    exit()

# Load the posts and votes data
try:
    posts_df = pd.read_csv('reddit_showerthoughts.tsv', sep='\t', on_bad_lines='skip').dropna(subset=['title'])
    # Use the specified votes file
    votes_df = pd.read_csv('44_million_votes.txt', sep='\t')
except FileNotFoundError:
    print("Error: 'reddit_showerthoughts.tsv' or '44_million_vote.txt' not found.")
    exit()

print("Data loaded successfully.")
print("-" * 30)

--- 1. Loading all data ---
Data loaded successfully.
------------------------------


In [ ]:
# 2. Prepare Data for Evaluation
print("\n--- 2. Preparing data for evaluation ---")
# For a standard Precision/Recall evaluation, we define "relevant" items as upvoted items.
upvotes_df = votes_df[votes_df['VOTE'] == 'upvote']

# Create a reliable index for the embedding matrix
posts_df = posts_df.reset_index(drop=True)
posts_df['embedding_idx'] = posts_df.index

# Merge votes with posts to get post details and the new embedding_idx
posts_df.rename(columns={'submission_id': 'SUBMISSION_ID'}, inplace=True)
merged_df = pd.merge(upvotes_df, posts_df, on='SUBMISSION_ID')

# For meaningful evaluation, focus on users with a decent number of upvotes
user_vote_counts = merged_df['USERNAME'].value_counts()
active_users = user_vote_counts[user_vote_counts >= 10].index.tolist() # 10 upvotes is a good threshold
eval_df = merged_df[merged_df['USERNAME'].isin(active_users)]

print(f"Found {len(active_users)} active users (>= 10 upvotes) for evaluation.")
print("-" * 30)


--- 2. Preparing data for evaluation ---
Found 4628 active users (>= 10 upvotes) for evaluation.
------------------------------


In [ ]:
# 3. Standard Evaluation Function (MRR, Precision, Recall)
def evaluate_recommender(user_item_df, item_features_matrix, top_n=100):
    """
    Performs an offline evaluation using MRR, Precision@N and Recall@N.
    """
    user_groups = user_item_df.groupby('USERNAME')
    precisions = []
    recalls = []
    reciprocal_ranks = []

    print(f"Evaluating for {len(user_groups)} users...")
    for username, group in tqdm(user_groups):
        # Split user's upvoted items into a training set and a test set
        if len(group) < 5: # Need at least a few items to train and test
            continue

        test_size = max(1, int(len(group) * 0.2))
        train_items = group.sample(frac=1, random_state=42).iloc[test_size:]
        test_items = group.sample(frac=1, random_state=42).iloc[:test_size]

        train_indices = train_items['embedding_idx'].tolist()
        test_indices = set(test_items['embedding_idx'].tolist())

        # Build user profile from the training items
        user_profile = item_features_matrix[train_indices].mean(axis=0)
        if isinstance(user_profile, np.matrix):
            user_profile = np.asarray(user_profile)
        user_profile = user_profile.reshape(1, -1)

        # Generate recommendations
        similarity_scores = cosine_similarity(user_profile, item_features_matrix).flatten()
        similarity_scores[train_indices] = -1 # Exclude items from training set

        # Get all ranked indices (not just top N)
        ranked_indices = similarity_scores.argsort()[::-1]

        # --- Calculate Metrics ---
        # For Precision and Recall, we only consider the top N
        recommended_indices_top_n = set(ranked_indices[:top_n])
        hits = len(recommended_indices_top_n & test_indices)

        precision_at_n = hits / top_n
        recall_at_n = hits / len(test_indices)
        precisions.append(precision_at_n)
        recalls.append(recall_at_n)

        # For MRR, we find the rank of the *first* hit in the full ranked list
        rr = 0.0
        for rank, idx in enumerate(ranked_indices):
            if idx in test_indices:
                rr = 1 / (rank + 1)
                break # Found the first hit, stop searching
        reciprocal_ranks.append(rr)


    avg_precision = np.mean(precisions) if precisions else 0
    avg_recall = np.mean(recalls) if recalls else 0
    avg_mrr = np.mean(reciprocal_ranks) if reciprocal_ranks else 0
    return avg_mrr, avg_precision, avg_recall

In [ ]:
# 4. Run Evaluation for Both Models
print("\n--- 4. Running evaluations ---")
TOP_K = 100

# A) Evaluate LLM-based Recommender
print(f"\n--- Evaluating LLM Recommender (Top {TOP_K}) ---")
llm_mrr, llm_precision, llm_recall = evaluate_recommender(eval_df, combined_llm_embeddings, top_n=TOP_K)
print(f"\nLLM Model Results:")
print(f"  - Average Mean Reciprocal Rank (MRR): {llm_mrr:.4f}")
print(f"  - Average Precision@{TOP_K}: {llm_precision:.4f}")
print(f"  - Average Recall@{TOP_K}:    {llm_recall:.4f}")

# B) Evaluate TF-IDF-based Recommender
print(f"\n--- Evaluating TF-IDF Recommender (Top {TOP_K}) ---")
# First, create the TF-IDF matrix for all posts
posts_df['content'] = posts_df['title'] + ' ' + posts_df['selftext'].fillna('')
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_df=0.95, min_df=2)
tfidf_matrix = tfidf_vectorizer.fit_transform(posts_df['content'])

tfidf_mrr, tfidf_precision, tfidf_recall = evaluate_recommender(eval_df, tfidf_matrix, top_n=TOP_K)
print(f"\nTF-IDF Model Results:")
print(f"  - Average Mean Reciprocal Rank (MRR): {tfidf_mrr:.4f}")
print(f"  - Average Precision@{TOP_K}: {tfidf_precision:.4f}")
print(f"  - Average Recall@{TOP_K}:    {tfidf_recall:.4f}")


--- 4. Running evaluations ---

--- Evaluating LLM Recommender (Top 100) ---
Evaluating for 4628 users...


100%|██████████| 4628/4628 [10:56<00:00,  7.05it/s]



LLM Model Results:
  - Average Mean Reciprocal Rank (MRR): 0.0021
  - Average Precision@100: 0.0002
  - Average Recall@100:    0.0030

--- Evaluating TF-IDF Recommender (Top 100) ---
Evaluating for 4628 users...


100%|██████████| 4628/4628 [00:39<00:00, 116.17it/s]



TF-IDF Model Results:
  - Average Mean Reciprocal Rank (MRR): 0.0060
  - Average Precision@100: 0.0002
  - Average Recall@100:    0.0049


# 5 Vote Prediction

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [2]:
# --- 1. Load All Necessary Data ---
print("--- 1. Loading all data ---")
# Load the pre-computed LLM embeddings (these are best for capturing semantic taste)
try:
    embeddings_data = np.load('showerthoughts_combined_embeddings.npz')
    # title_embeddings = embeddings_data['title_embed']
    # content_embeddings = embeddings_data['content_embed']
    # combined_llm_embeddings = (0.6 * title_embeddings) + (0.4 * content_embeddings)
    embeddings_text = embeddings_data['combined_embed']
    embeddings_subid = embeddings_data['ids']

    # embeddings=pd.DataFrame({'SUBMISSION_ID':subid_embeddings,'combined_embed':combined_llm_embeddings})
    # combined_llm_embeddings[:10]
    # embeddings.head(10)


except FileNotFoundError:
    print("Error: 'post_embeddings.npz' not found. Please run the embedding generation script first.")
    # exit()

# Load the posts and votes data
try:
    posts_df = pd.read_csv('reddit_showerthoughts.tsv', sep='\t', on_bad_lines='skip').dropna(subset=['title']).drop(columns=['index'])
    votes_df = pd.read_csv('44_million_votes.txt', sep='\t')
except FileNotFoundError:
    print("Error: 'reddit_showerthoughts.tsv' or '44_million_vote.txt' not found.")
    # exit()

print("Data loaded successfully.")
print("-" * 30)

--- 1. Loading all data ---
Data loaded successfully.
------------------------------


In [11]:
# --- 2. Prepare Data and Select a User ---
print("\n--- 2. Preparing data and selecting a user ---")
# Filter for upvotes and downvotes
votes_df = votes_df[votes_df['VOTE'].isin(['upvote', 'downvote'])]

# Create a reliable index for the embedding matrix
# posts_df = posts_df.reset_index(drop=True)
# posts_df['embedding_idx'] = posts_df['submission_id']
# posts_df['embedding_idx']
votes_df = votes_df.reset_index(drop=True)
votes_df['embedding_idx'] = votes_df.index
# votes_df['embedding_idx']
# Merge votes with posts
posts_df.rename(columns={'submission_id': 'SUBMISSION_ID'}, inplace=True)
merged_df = pd.merge(votes_df, posts_df, on='SUBMISSION_ID')

# Find a user with a good mix of upvotes and downvotes for a good demonstration
user_vote_counts = merged_df.groupby('USERNAME')['VOTE'].value_counts().unstack().fillna(0)
active_users = user_vote_counts[(user_vote_counts['upvote'] >= 40) & (user_vote_counts['downvote'] >= 10)]
if active_users.empty:
    print("Could not find a user with at least 10 upvotes and 10 downvotes.")
    exit()
target_user = active_users.index[10]
user_history = merged_df[merged_df['USERNAME'] == target_user]

print(f"Selected target user: {target_user}")
print(f"User has {len(user_history)} total votes.")
print("-" * 30)
len(active_users)


--- 2. Preparing data and selecting a user ---
Selected target user: Adnan_Targaryen
User has 201 total votes.
------------------------------


304

In [12]:
# --- 3. Split User History and Build User Profiles ---
print("\n--- 3. Splitting data and building user taste profiles ---")
# Split the user's history into a training set (to build profiles) and a test set (to evaluate)

train_df, test_df = train_test_split(user_history, test_size=0.3, random_state=42, stratify=user_history['VOTE'])
embeding_train_indices_num=[i for i,x in enumerate(embeddings_subid) if x in train_df[train_df['VOTE'] == 'upvote']['SUBMISSION_ID'].tolist()+train_df[train_df['VOTE'] == 'downvote']['SUBMISSION_ID'].tolist()]
train_df=train_df[train_df['SUBMISSION_ID'].isin(embeddings_subid[embeding_train_indices_num])]
embeding_test_indices_num=[i for i,x in enumerate(embeddings_subid) if x in test_df[test_df['VOTE'] == 'upvote']['SUBMISSION_ID'].tolist()+test_df[test_df['VOTE'] == 'downvote']['SUBMISSION_ID'].tolist()]
test_df=test_df[test_df['SUBMISSION_ID'].isin(embeddings_subid[embeding_test_indices_num])]

# train_df[:10]
# len(train_df)

# Separate the user's training history into upvoted and downvoted posts
upvoted_train_indices = train_df[train_df['VOTE'] == 'upvote']['SUBMISSION_ID'].tolist()
# upvoted_train_indices=[x for x in upvoted_train_indices if x in embeddings_subid[embeding_indices_num]]
downvoted_train_indices = train_df[train_df['VOTE'] == 'downvote']['SUBMISSION_ID'].tolist()
# downvoted_train_indices=[x for x in downvoted_train_indices if x in embeddings_subid[embeding_indices_num]]
# test_df
upvoted_test_indices = test_df[test_df['VOTE'] == 'upvote']['SUBMISSION_ID'].tolist()
# upvoted_test_indices=[x for x in upvoted_test_indices if x in embeddings_subid[embeding_indices_num]]
downvoted_test_indices = test_df[test_df['VOTE'] == 'downvote']['SUBMISSION_ID'].tolist()
# downvoted_test_indices=[x for x in downvoted_test_indices if x in embeddings_subid[embeding_indices_num]]

# downvoted_train_indices

embedings_upvoted_train_indices=[i for i,x in enumerate(embeddings_subid) if x in upvoted_train_indices]
embedings_downvoted_train_indices=[i for i,x in enumerate(embeddings_subid) if x in downvoted_train_indices]


# Create two taste profiles: one for what they like, one for what they dislike
upvote_profile = embeddings_text[embedings_upvoted_train_indices].mean(axis=0).reshape(1, -1)
downvote_profile = embeddings_text[embedings_downvoted_train_indices].mean(axis=0).reshape(1, -1)

print("Successfully built 'upvote' and 'downvote' profiles for the user.")
print("-" * 30)
len(test_df)


--- 3. Splitting data and building user taste profiles ---
Successfully built 'upvote' and 'downvote' profiles for the user.
------------------------------


61

In [13]:
# --- 4. Create Feature Set for the Classifier ---
print("\n--- 4. Creating features for the classifier ---")
def create_features(item_indices, up_profile, down_profile, embeddings):
    """Calculates similarity features for a set of items."""
    item_vectors = embeddings[item_indices]
    
    # Feature 1: Similarity to what the user likes
    sim_to_upvotes = cosine_similarity(item_vectors, up_profile).flatten()
    
    # Feature 2: Similarity to what the user dislikes
    sim_to_downvotes = cosine_similarity(item_vectors, down_profile).flatten()
    
    # Combine into a feature matrix
    return np.vstack([sim_to_upvotes, sim_to_downvotes]).T

# Create features for the training set
X_train = create_features(embeding_train_indices_num, upvote_profile, downvote_profile, embeddings_text)
y_train = (train_df['VOTE'] == 'upvote').astype(int) # Target: 1 for upvote, 0 for downvote
# y_train = train_df[train_df['VOTE'] == 'upvote']['VOTE'].tolist()  # Target: 1 for upvote, 0 for downvote
# train_indices_numerical=[i for i,_ in enumerate(train_df['VOTE']) if train_df.iloc[i]['SUBMISSION_ID'] in upvoted_train_indices+downvoted_train_indices]
# y_train=[1 if x in train_df['VOTE'] else 0 for i,x in enumerate(train_df[]['VOTE'])]
# numerical_indices[:10]


# Create features for the test set
# embeding_indices_num=[i for i,x in enumerate(embeddings_subid) if x in upvoted_test_indices+downvoted_test_indices]
X_test = create_features(embeding_test_indices_num, upvote_profile, downvote_profile, embeddings_text)
y_test = (test_df['VOTE'] == 'upvote').astype(int)

print(f"Created training features with shape: {X_train.shape}")
print(f"Created test features with shape: {X_test.shape}")
print("-" * 30)


--- 4. Creating features for the classifier ---
Created training features with shape: (136, 2)
Created test features with shape: (61, 2)
------------------------------


In [14]:
# --- 5. Train Classifier and Make Predictions ---
print("\n--- 5. Training model and making predictions ---")
# Train a simple but effective Logistic Regression model
classifier = LogisticRegression(random_state=42, class_weight='balanced')
classifier.fit(X_train, y_train)

# Make predictions on the test set
predictions = classifier.predict(X_test)
prediction_probs = classifier.predict_proba(X_test)[:, 1] # Probability of upvote

print("Model trained. Predictions made on the test set.")
print("-" * 30)


--- 5. Training model and making predictions ---
Model trained. Predictions made on the test set.
------------------------------


In [15]:
# --- 6. Display Results ---
print("\n--- 6. Prediction Results ---")
# Add predictions to the test dataframe for easy viewing
test_df['predicted_vote'] = ['upvote' if p == 1 else 'downvote' for p in predictions]
test_df['upvote_probability'] = prediction_probs

# Display a sample of the results
print("Sample of predictions vs actual votes:")
for _, row in test_df.iterrows():
    print(f"\nPost: '{row['title']}'")
    print(f"  - Prediction: {row['predicted_vote']} (Prob: {row['upvote_probability']:.2f})")
    print(f"  - Actual Vote: {row['VOTE']}")

# Print a full classification report
print("\n\n--- Overall Model Performance ---")
print(classification_report(y_test, predictions, target_names=['downvote', 'upvote']))


--- 6. Prediction Results ---
Sample of predictions vs actual votes:

Post: 'Telemarketers have basically ruined the telephone as a tool for contacting people quickly because no one bothers to answer it anymore.'
  - Prediction: upvote (Prob: 0.50)
  - Actual Vote: upvote

Post: 'If you think you'll regret something in the morning, be a problem solver & sleep until noon.'
  - Prediction: downvote (Prob: 0.48)
  - Actual Vote: downvote

Post: 'If I cant find what Im looking for on the first page of Google, Id rather reword my search than continue on to page two.'
  - Prediction: downvote (Prob: 0.45)
  - Actual Vote: downvote

Post: 'Every year you pass your birthday and know you were born that day but every year you pass your death date and have no clue'
  - Prediction: upvote (Prob: 0.51)
  - Actual Vote: downvote

Post: 'Any salad can be a Caesar salad if you stab it hard enough'
  - Prediction: upvote (Prob: 0.51)
  - Actual Vote: downvote

Post: 'What is wrong with you? and "What 